<p align="center"><a href="https://www.plus2net.com/python/file-tell.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python file.tell(): Tracking Stream Pointer Byte Positions Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/file-tell.php](https://www.plus2net.com/python/file-tell.php)

Run cells in order in Google Colab. All file cursor tracking concepts, inspecting offsets after reading and writing, pairing `tell()` with `seek()` for bookmarking and restoration, calculating stream sizes dynamically, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Setup: Creating Sample Files for Cursor Tracking
Generate text and binary test files.

In [ ]:
sample_data = """Header: System Log Record
Line 1: Worker initialized
Line 2: Task completed
Line 3: Finished pipeline
"""

with open("demo_tell.txt", "w", encoding="utf-8") as f:
    f.write(sample_data)

print("Created demo_tell.txt successfully.")


### 2. Inspecting Cursor Position Before and After Reading
`tell()` reports 0 at start and advances as lines are consumed.

In [ ]:
with open("demo_tell.txt", "r", encoding="utf-8") as fob:
    print("Initial offset:", fob.tell())  # 0
    
    line1 = fob.readline()
    print("Read Line 1:  ", repr(line1.strip()))
    print("Offset after 1:", fob.tell())
    
    line2 = fob.readline()
    print("Read Line 2:  ", repr(line2.strip()))
    print("Offset after 2:", fob.tell())


### 3. The Bookmark-and-Restore Pattern with seek() and tell()
Save a cursor position, inspect ahead, and rewind back to the bookmark.

In [ ]:
with open("demo_tell.txt", "r", encoding="utf-8") as stream:
    hdr = stream.readline()
    
    # Record checkpoint bookmark
    bookmark = stream.tell()
    print("Saved bookmark at offset:", bookmark)
    
    # Read ahead
    peek = stream.readline()
    print("Peeked line: ", repr(peek.strip()))
    
    # Restore cursor to bookmark
    stream.seek(bookmark)
    print("Restored offset:", stream.tell())
    re_read = stream.readline()
    print("Re-read line:", repr(re_read.strip()))


### 4. Dynamic Stream Size Calculation via seek(0, 2) + tell()
Determine file size directly on an open stream without filesystem stat metadata.

In [ ]:
import os

def get_stream_size(filepath: str) -> int:
    with open(filepath, "rb") as stream:
        stream.seek(0, os.SEEK_END)
        return stream.tell()

calculated_size = get_stream_size("demo_tell.txt")
os_stat_size = os.path.getsize("demo_tell.txt")
print("Calculated via tell():", calculated_size)
print("OS stat reported:     ", os_stat_size)
print("Match?", calculated_size == os_stat_size)


### 5. Resumable Chunk Reader with tell() Offsets
Simulate an ETL worker that records byte offsets to resume after interruptions.

In [ ]:
def process_chunks_with_offsets(path: str, chunk_size: int = 24):
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            current_offset = f.tell()
            print(f"Processed {len(chunk)} bytes -> Cursor now at byte {current_offset}")

process_chunks_with_offsets("demo_tell.txt")


### 6. Self-Checking Verification Suite
Run unit assertions verifying `tell()` offset tracking and bookmark restoration.

In [ ]:
class TellVerifier:
    @staticmethod
    def get_size(path: str) -> int:
        with open(path, "rb") as stream:
            stream.seek(0, 2)
            return stream.tell()

v = TellVerifier()
assert v.get_size("demo_tell.txt") == os.path.getsize("demo_tell.txt")

with open("demo_tell.txt", "r", encoding="utf-8") as f:
    assert f.tell() == 0
    f.readline()
    bm = f.tell()
    assert bm > 0
    l2 = f.readline()
    f.seek(bm)
    assert f.readline() == l2

print("All file.tell() unit assertions passed successfully! 100% verified.")
